# Support Vector Regression (SVR)

SVR fits a smooth curve inside a tube. A point inside the tube costs nothing, so only the points outside it pull on the curve. Here it predicts **mpg** from **horsepower**.

This notebook goes with the **Support Vector Regression** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

**The steps:** load the data → split → scale x and y → fit → check the result → try the three settings.

## 1. Load the data

Each row is a car from the late 1970s and early 1980s. We predict `mpg` from `horsepower`. A few cars have no horsepower listed, so we drop them first.

The data is the [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) data set from the UCI Machine Learning Repository (CC BY 4.0). See `datasets/README.md` for what we changed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/mpg.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

In [ ]:
dataset = dataset.dropna(subset=['horsepower'])
X = dataset[['horsepower']].values
y = dataset['mpg'].values.reshape(-1, 1)   # 2-D for the scaler
print(X.shape, 'cars')

## 2. Split into train and test

The model learns from the **training** cars (75%). We hold back the other 25% and only use them at the end. The test score shows how the model does on cars it has never seen. These are the ringed points on the site's boards.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(len(X_train), 'train,', len(X_test), 'test')

## 3. Scale x and y

SVR works with distances and with a tube of fixed width, so both the input **and the target** need scaling. We fit each scaler on the **training** data only, then use it on both sets. This is the one regression model in this course where y is scaled too.

In [ ]:
sc_X, sc_y = StandardScaler(), StandardScaler()
X_train_s = sc_X.fit_transform(X_train)
X_test_s = sc_X.transform(X_test)
y_train_s = sc_y.fit_transform(y_train).ravel()

## 4. Fit the model

The settings you will meet on the site: `epsilon` is the tube half-width, `C` is how hard a point outside the tube can pull, and `gamma` is how bendy the curve may be.

In [ ]:
regressor = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')
regressor.fit(X_train_s, y_train_s)
print('support vectors:', len(regressor.support_), 'of', len(X_train_s), 'training cars')

def predict(X_raw):
    # scale x, predict, then turn the answer back into real mpg
    return sc_y.inverse_transform(regressor.predict(sc_X.transform(X_raw)).reshape(-1, 1)).ravel()

## 4. Check the result

**R²** says how much of the ups and downs in the data the model explains. 1 is perfect, 0 is no better than always guessing the average. The number that matters is the **test** R². The last line shows the typical miss in miles per gallon.

In [ ]:
y_pred_train = predict(X_train)
y_pred_test = predict(X_test)
print('Train R²:', round(r2_score(y_train, y_pred_train), 3))
print('Test R²: ', round(r2_score(y_test, y_pred_test), 3))
print('Test error on average:', round(mean_absolute_error(y_test, y_pred_test), 2), 'mpg')

In [ ]:
grid = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
plt.figure(figsize=(7, 4.5))
plt.scatter(X_train, y_train, color='#5b7cfa', label='train')
plt.scatter(X_test, y_test, facecolors='white', edgecolors='#5b7cfa', linewidths=1.8, label='test')
line = predict(grid)
plt.plot(grid, line, color='black', label='SVR curve')
tube = regressor.epsilon * sc_y.scale_[0]          # tube half-width back in mpg
plt.fill_between(grid.ravel(), line - tube, line + tube, color='#d6336c', alpha=0.15, label='epsilon tube')
plt.xlabel('horsepower'); plt.ylabel('mpg'); plt.legend()
plt.show()

## Try this

1. Set `epsilon=0.5`. The tube widens: how many support vectors are left, and what happens to the curve?
2. Set `gamma=5`. The curve gets wiggly. Compare train and test R².
3. Try `C=0.1` and `C=1000`. Which one is closer to a straight line?